# Notebook 4: 최종 평가

## 목표
SlideAudit 라벨 데이터로 전체 파이프라인 AUC 측정.
Isolation Forest(기존) vs 새 파이프라인(CNN + HMM) 정량 비교.

## 평가 지표
- **AUC-ROC**: 이상 탐지 정확도
- **Precision / Recall @ threshold**
- **Ablation Study**: IF만 / HMM만 / 결합
- **가중치 그리드 탐색**: 최적 α 찾기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'
SLIDEAUDIT_DIR = f'{BASE_DIR}/slideaudit'

In [ ]:
!pip install -q timm hmmlearn datasets
print('설치 완료')

## 1. SlideAudit 데이터셋 로드

GitHub: https://github.com/WeichenSunAlex/UIST_2025
2,400장 슬라이드, 폰트/색상/레이아웃 결함 라벨

In [ ]:
import os
import subprocess

os.makedirs(SLIDEAUDIT_DIR, exist_ok=True)

if not os.path.exists(f'{SLIDEAUDIT_DIR}/UIST_2025'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/WeichenSunAlex/UIST_2025',
         f'{SLIDEAUDIT_DIR}/UIST_2025'],
        check=True
    )
    print('SlideAudit 클론 완료')
else:
    print('SlideAudit 이미 존재')

for root, dirs, files in os.walk(f'{SLIDEAUDIT_DIR}/UIST_2025'):
    depth = root.replace(f'{SLIDEAUDIT_DIR}/UIST_2025', '').count(os.sep)
    if depth < 2:
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        if depth == 1:
            for f in files[:5]:
                print(f'{indent}  {f}')

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

label_files = list(Path(f'{SLIDEAUDIT_DIR}/UIST_2025').rglob('*.json')) + \
              list(Path(f'{SLIDEAUDIT_DIR}/UIST_2025').rglob('*.csv'))

print('발견된 라벨 파일:')
for f in label_files:
    print(f'  {f}')

In [ ]:
label_file = label_files[0]

if label_file.suffix == '.json':
    with open(label_file) as f:
        raw = json.load(f)
    print('JSON 구조 (첫 항목):')
    if isinstance(raw, list):
        print(json.dumps(raw[0], indent=2, ensure_ascii=False))
        slideaudit_df = pd.DataFrame(raw)
    else:
        print(json.dumps(list(raw.items())[:3], indent=2, ensure_ascii=False))
elif label_file.suffix == '.csv':
    slideaudit_df = pd.read_csv(label_file)
    print(slideaudit_df.head())
    print(slideaudit_df.columns.tolist())

print(f'\nSlideAudit 샘플 수: {len(slideaudit_df)}')

## 2. 파이프라인 구성

- CNN 모델 로드 (slide_norm.json 정규화 통계 사용)
- PCA 모델 로드 (step2 저장)
- 전역 IsolationForest 학습 (PCA 임베딩 기반)
- HMM 모델 로드

In [ ]:
import torch
import timm
import torch.nn as nn
import pickle
from torchvision import transforms
from PIL import Image
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']
NUM_CLASSES = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class SlideRoleClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=False,
                                          num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feat_dim, 256),
            nn.ReLU(), nn.Dropout(dropout * 0.5), nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def extract_features(self, x):
        return self.backbone(x)


# CNN 모델 로드
cnn_model = SlideRoleClassifier().to(device)
ckpt = torch.load(f'{MODELS_DIR}/role_classifier_best.pt', map_location=device)
cnn_model.load_state_dict(ckpt['model_state_dict'])
cnn_model.eval()
print(f'CNN 모델 로드 완료 (val_acc={ckpt["val_acc"]:.4f})')

# 정규화 통계 로드 (NB02와 동일한 값 — 임베딩 분포 일치)
with open(f'{MODELS_DIR}/slide_norm.json') as f:
    norm = json.load(f)
SLIDE_MEAN = norm['mean']
SLIDE_STD  = norm['std']

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=SLIDE_MEAN, std=SLIDE_STD),
])
print(f'정규화 통계 로드 완료: mean={[round(v,4) for v in SLIDE_MEAN]}')

# HMM + 임계값 로드
with open(f'{MODELS_DIR}/hmm_model.pkl', 'rb') as f:
    hmm_model = pickle.load(f)
print('HMM 모델 로드 완료')

with open(f'{MODELS_DIR}/hmm_thresholds.json') as f:
    thresholds = json.load(f)
print(f'임계값 로드: method={thresholds["method"]}')

In [ ]:
# PCA 모델 로드 + 전역 Isolation Forest 학습 (PCA 임베딩 기반)
# IF는 덱마다 fit하지 않고 전역으로 한 번만 학습 — 통계적 안정성 확보
with open(f'{MODELS_DIR}/pca_model.pkl', 'rb') as f:
    pca_bundle = pickle.load(f)
pca_scaler = pca_bundle['scaler']
pca_model  = pca_bundle['pca']

all_embeddings_pca = np.load(f'{LABELS_DIR}/embeddings_pca.npy')
global_iso = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
global_iso.fit(all_embeddings_pca)

with open(f'{MODELS_DIR}/isolation_forest.pkl', 'wb') as f:
    pickle.dump(global_iso, f)
print(f'전역 IF 학습 완료: {all_embeddings_pca.shape} (PCA 축소 임베딩)')

## 3. 배치 추론 + 이상 점수 계산

이미지 하나씩 처리하는 루프 → DataLoader 배치 추론으로 교체.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


class InferenceDataset(Dataset):
    def __init__(self, image_paths: list, transform):
        self.paths     = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.paths[idx]).convert('RGB')), self.paths[idx]


def extract_embeddings_batch(
    image_paths: list,
    model: nn.Module,
    transform,
    batch_size: int = 64,
) -> tuple:
    loader = DataLoader(InferenceDataset(image_paths, transform),
                        batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    all_emb, all_roles = [], []
    model.eval()
    with torch.no_grad():
        for imgs, _ in loader:
            emb   = model.extract_features(imgs.to(device))
            roles = model.classifier(emb).argmax(dim=1)
            all_emb.append(emb.cpu().numpy())
            all_roles.append(roles.cpu().numpy())
    return np.concatenate(all_emb), np.concatenate(all_roles)


def compute_anomaly_score(image_paths: list) -> dict:
    """
    한 덱의 슬라이드 이미지 리스트를 받아 이상 점수 반환.
    compute_anomaly_score 내에서 IsolationForest().fit()을 호출하지 않는다.
    반드시 전역 global_iso를 사용한다 — 덱 단위 fit은 샘플 수 부족으로 무의미하다.
    """
    if len(image_paths) < 2:
        return {'slide_scores': [0.5] * len(image_paths),
                'pred_roles':   [2]   * len(image_paths),
                'deck_structural_score': 0.5,
                'if_scores':    [0.5] * len(image_paths),
                'hmm_score':    0.5}

    embeddings_np, pred_roles = extract_embeddings_batch(image_paths, cnn_model, transform)

    # PCA 변환 후 전역 IF — 1536차원 직접 입력 금지 (차원의 저주)
    emb_pca = pca_model.transform(pca_scaler.transform(embeddings_np))
    raw     = global_iso.decision_function(emb_pca)
    s_min, s_max = raw.min(), raw.max()
    if_scores = 1.0 - (raw - s_min) / (s_max - s_min + 1e-8)

    # HMM 구조 점수
    seq       = pred_roles.reshape(-1, 1)
    ll        = hmm_model.score(seq) / len(seq)
    z         = (thresholds['mean'] - ll) / (thresholds['std'] + 1e-8)
    hmm_score = float(np.clip(z / 3.0, 0, 1))

    return {
        'slide_scores': (0.7 * if_scores + 0.3 * hmm_score).tolist(),
        'pred_roles':   pred_roles.tolist(),
        'deck_structural_score': hmm_score,
        'if_scores':    if_scores.tolist(),
        'hmm_score':    hmm_score,
    }


print('이상 점수 계산 함수 준비 완료')

## 4. SlideAudit로 AUC 측정

In [ ]:
# 덱별 결과를 리스트로 보존 — Ablation, 가중치 탐색 공유
all_results  = []
valid_groups = []

for deck_id, group in tqdm(slideaudit_df.groupby('deck_id'), desc='AUC 계산'):
    if 'slide_idx' in group.columns:
        group = group.sort_values('slide_idx')
    image_paths = group['image_path'].tolist()
    valid_pairs = [(p, row) for p, (_, row) in zip(image_paths, group.iterrows())
                   if Path(p).exists()]
    if len(valid_pairs) < 2:
        continue
    valid_paths = [p for p, _ in valid_pairs]
    valid_group = group[group['image_path'].isin(valid_paths)]
    result      = compute_anomaly_score(valid_paths)
    all_results.append(result)
    valid_groups.append((deck_id, valid_group))

all_scores_arr = np.array([s for r in all_results for s in r['slide_scores']])
all_labels_arr = np.array([l for _, g in valid_groups for l in g['has_defect'].astype(int).tolist()])

auc_new = roc_auc_score(all_labels_arr, all_scores_arr)
print(f'평가 슬라이드 수: {len(all_scores_arr)}')
print(f'이상 슬라이드 비율: {all_labels_arr.mean():.2%}')
print(f'\n새 파이프라인 AUC: {auc_new:.4f}')

## 5. 베이스라인 (전역 IF on raw embeddings)

In [ ]:
# 베이스라인: PCA 없는 전역 IF (원본 1536차원) — 차원의 저주를 보여주기 위한 비교
from sklearn.ensemble import IsolationForest as IF_baseline

embeddings_all = []
for img_path in tqdm(slideaudit_df['image_path'].tolist()[:len(all_scores_arr)], desc='베이스라인 임베딩'):
    if not Path(img_path).exists():
        embeddings_all.append(np.zeros(1536))
        continue
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = cnn_model.extract_features(tensor).squeeze(0).cpu().numpy()
    embeddings_all.append(emb)

emb_matrix = np.array(embeddings_all)
iso_baseline = IF_baseline(n_estimators=100, contamination=0.2, random_state=42)
iso_baseline.fit(emb_matrix)
baseline_raw = iso_baseline.decision_function(emb_matrix)
s_min, s_max = baseline_raw.min(), baseline_raw.max()
baseline_scores = 1.0 - (baseline_raw - s_min) / (s_max - s_min + 1e-8)

auc_baseline = roc_auc_score(all_labels_arr, baseline_scores[:len(all_labels_arr)])

print(f'=== 기본 비교 ===')
print(f'기존 Isolation Forest (원본 임베딩) AUC: {auc_baseline:.4f}')
print(f'새 파이프라인 AUC:                       {auc_new:.4f}')
print(f'개선폭:                                  +{auc_new - auc_baseline:.4f}')

## 6. Ablation Study

In [ ]:
# IF만 — 슬라이드 단위 flatten
if_scores_flat = np.array([s for r in all_results for s in r['if_scores']])
auc_if_only    = roc_auc_score(all_labels_arr, if_scores_flat)

# HMM만 — 덱 점수를 슬라이드 수만큼 broadcast
hmm_flat = np.array([r['hmm_score'] for r, (_, g) in zip(all_results, valid_groups)
                     for _ in range(len(g))])
auc_hmm_only = roc_auc_score(all_labels_arr, hmm_flat)

print('=== Ablation Study ===')
print(f'IF만:                  AUC = {auc_if_only:.4f}')
print(f'HMM만:                 AUC = {auc_hmm_only:.4f}')
print(f'IF(0.7) + HMM(0.3):   AUC = {auc_new:.4f}')

## 7. 가중치 그리드 탐색

In [ ]:
alphas       = np.linspace(0.0, 1.0, 21)
auc_by_alpha = []

for alpha in alphas:
    combined = np.array([
        alpha * s + (1 - alpha) * r['hmm_score']
        for r in all_results for s in r['if_scores']
    ])
    auc_by_alpha.append(roc_auc_score(all_labels_arr, combined))

best_alpha     = alphas[np.argmax(auc_by_alpha)]
best_auc_alpha = max(auc_by_alpha)
print(f'최적 IF 가중치: α={best_alpha:.2f} (AUC={best_auc_alpha:.4f})')

plt.figure(figsize=(8, 4))
plt.plot(alphas, auc_by_alpha, 'o-', color='steelblue')
plt.axvline(best_alpha, color='red', linestyle='--', label=f'최적 α={best_alpha:.2f}')
plt.xlabel('α (IF 가중치)')
plt.ylabel('AUC')
plt.title('IF-HMM 가중치 그리드 탐색')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/weight_grid_search.png', dpi=120)
plt.show()

## 8. ROC 커브 + 최종 결과 저장

In [ ]:
from sklearn.metrics import roc_curve

fpr_new, tpr_new, _ = roc_curve(all_labels_arr, all_scores_arr)
fpr_base, tpr_base, _ = roc_curve(all_labels_arr, baseline_scores[:len(all_labels_arr)])

plt.figure(figsize=(8, 6))
plt.plot(fpr_new, tpr_new, color='steelblue',
         label=f'새 파이프라인 (AUC={auc_new:.4f})', linewidth=2)
plt.plot(fpr_base, tpr_base, color='tomato', linestyle='--',
         label=f'IF 베이스라인 (AUC={auc_baseline:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k:', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve 비교')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/roc_comparison.png', dpi=120)
plt.show()

final_results = {
    'baseline_auc':               float(auc_baseline),
    'new_pipeline_auc':           float(auc_new),
    'improvement':                float(auc_new - auc_baseline),
    'best_alpha':                 float(best_alpha),
    'best_auc_with_optimal_alpha': float(best_auc_alpha),
    'n_eval_slides':              int(len(all_labels_arr)),
    'defect_ratio':               float(all_labels_arr.mean()),
    'ablation': {
        'if_only':          float(auc_if_only),
        'hmm_only':         float(auc_hmm_only),
        'combined_0.7_0.3': float(auc_new),
        'combined_optimal': float(best_auc_alpha),
    },
}
with open(f'{MODELS_DIR}/final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)
print(json.dumps(final_results, indent=2))

In [ ]:
print('\n=== AUC 해석 ===')
if auc_new >= 0.75:
    print(f'✓ 우수 (AUC={auc_new:.4f}): 파이프라인이 SlideAudit 결함과 높은 상관')
elif auc_new >= 0.65:
    print(f'△ 보통 (AUC={auc_new:.4f}): 일부 결함이 구조/비주얼 이상과 겹침')
else:
    print(f'▲ 참고 (AUC={auc_new:.4f}): SlideAudit는 디자인 결함, 우리 모델은 구조/비주얼 이상')
    print(f'  IF만={auc_if_only:.4f} / HMM만={auc_hmm_only:.4f} 로 컴포넌트별 기여 확인')

print('\n=== Notebook 4 완료 ===')
print('모든 모델 파일이 Google Drive에 저장되었습니다.')
print('다음 단계: 백엔드 통합 (backend/app/pipeline/ 교체)')